In [1]:
sql_query = "SELECT sp.id, sp.email, sp.full_name, sp.phone, sp.specialty, sp.license_number, sp.years_of_experience, sp.is_active, sp.hire_date, sp.termination_date, sp.timezone, sp.created_at, sp.updated_at FROM service_providers sp LIMIT 10;"
def execute_sql_query(sql_query):
    query_upper = sql_query.strip().upper()
    if not query_upper.startswith("SELECT"):
        return "ERROR: Only SELECT queries are allowed for security reasons."

    # Check for dangerous keywords
    dangerous_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "CREATE", "TRUNCATE"]
    if any(keyword in query_upper for keyword in dangerous_keywords):
        return "ERROR: Query contains forbidden operations. Only SELECT queries allowed."


In [2]:
execute_sql_query(sql_query)

'ERROR: Query contains forbidden operations. Only SELECT queries allowed.'

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["conversation_db"]
collection = db["conversations"]

for doc in collection.find():
    print(doc)


In [2]:
docs = list(collection.find())
print(docs)


[]


In [5]:

"""
Base model for SQLAlchemy declarative base
"""
from sqlalchemy.orm import DeclarativeBase

class Base(DeclarativeBase):
    """Base class for all SQLAlchemy models"""
    pass
"""
SQLAlchemy Models for Dental Appointment Booking System
"""
from sqlalchemy import Column, Integer, String, Boolean, DateTime, Time, Date, ForeignKey, Enum, Text, Numeric
from sqlalchemy.orm import relationship
from sqlalchemy.sql import func
from datetime import datetime
import enum
import uuid
from sqlalchemy.dialects.postgresql import UUID
from sqlalchemy import (
    Column, Date, DateTime, Boolean, Numeric, ForeignKey, func, UniqueConstraint
)
# Import Base from models.base
# from .base import Base
# ============================================
# 1. USERS/CLIENTS TABLE
# ============================================
class User(Base):
    """Patient/Client table for dental appointment booking"""
    __tablename__ = "users"

    id = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    email = Column(String(255), unique=True, index=True, nullable=False)
    password_hash = Column(String(255), nullable=False)
    full_name = Column(String(255), nullable=False)
    phone = Column(String(20), nullable=False)

    # Dental-specific fields
    insurance_provider = Column(String(100))
    insurance_policy_number = Column(String(100))
    date_of_birth = Column(Date, nullable=False)

    # Medical history
    allergies = Column(Text)
    medical_conditions = Column(Text)
    current_medications = Column(Text)

    # Address information
    street_address = Column(String(255))
    city = Column(String(100))
    state = Column(String(50))
    zip_code = Column(String(10))

    # Timezone for appointment scheduling
    timezone = Column(String(50), default="America/Chicago", nullable=False)

    # Timestamps
    created_at = Column(DateTime(timezone=True), server_default=func.now())
    updated_at = Column(DateTime(timezone=True), onupdate=func.now())

    # Relationships
    appointments = relationship("Appointment", back_populates="user", cascade="all, delete-orphan")


In [6]:
from typing import Optional, List, Dict, Any
from sqlalchemy import select, update, delete
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy.orm import selectinload
from datetime import datetime
from uuid import UUID
from fastapi import HTTPException
# from ..models.dental_appointment_models import User

class UserService:
    """Service for handling user database operations"""

    @staticmethod
    async def get_user_by_id(db: AsyncSession, user_id: str) -> Optional[User]:
        """Get a single user by ID"""
        try:
            result = await db.execute(
                select(User)
                .where(User.id == user_id)
                .options(selectinload('*'))
            )
            return result.scalar_one_or_none()
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Error retrieving user: {str(e)}")


In [ ]:
user_manager = UserService()


from appointment-scheduler.backend.app.database import get_db
db = get_db()
user = await user_manager.get_user_by_id(db, str(userid))
if not user:
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="User not found"
    )
print(f"the user is {user}")